### Optical Flow

#### 1. 什么是 Optical Flow？

Optical Flow（光流）描述图像中像素外观随时间变化所呈现出的二维运动。

给定连续两帧图像：

- 第一帧：$I_1(x,y)$
- 第二帧：$I_2(x,y)$

光流为第一帧中的每个像素估计一个二维位移向量：

$$
\mathbf{f}(x,y)
=
\begin{bmatrix}
u(x,y)\\
v(x,y)
\end{bmatrix}
$$

其中：

- $u(x,y)$：水平方向的位移。
- $v(x,y)$：垂直方向的位移。
- 图像坐标通常以左上角为原点，$x$ 向右增加，$y$ 向下增加。

如果一个像素从第一帧的 $(x,y)$ 移动到第二帧的 $(x+u,y+v)$，那么对应的光流可以写为：

$$
I_1(x,y)
\longrightarrow
I_2(x+u,y+v)
$$

这里的光流表示图像平面上的二维位移，并不直接等于物体在三维世界中的真实运动。

**Sparse Optical Flow 与 Dense Optical Flow**

- Sparse Optical Flow：只估计部分特征点的运动。
- Dense Optical Flow：估计图像中几乎每个像素的运动。

RAFT 等现代光流方法主要用于稠密光流估计。

#### 2. 光流的基本假设：Brightness Constancy

传统光流方法通常从亮度恒常假设（Brightness Constancy Assumption）出发。

假设同一个场景点在相邻两帧中保持相同的亮度，则：

$$
I(x,y,t)
\approx
I(x+u,y+v,t+\Delta t)
$$

其中：

- $I(x,y,t)$：时刻 $t$ 的图像亮度。
- $u,v$：时间间隔 $\Delta t$ 内的图像位移。

该假设意味着：即使物体发生移动，跟随同一个场景点观察时，其图像亮度应近似保持不变。

**局限性：**

1. 光照变化会破坏亮度恒常性。
2. 镜面反射和高光可能使同一表面点的亮度改变。
3. 遮挡会使部分场景点在下一帧中不可见。
4. 无纹理区域缺乏足够的匹配信息。
5. 快速运动可能使相邻两帧中的位移过大，导致局部线性近似失效。

因此，亮度恒常性是一个有用的近似，而不是适用于所有场景的严格物理定律。

#### 3. Infinitesimal Optical Flow：微分光流约束

为了求解光流，可以将亮度恒常假设转化为关于图像梯度的方程。

从：

$$
I(x+u\Delta t,y+v\Delta t,t+\Delta t)
\approx I(x,y,t)
$$

出发，对左侧进行一阶 Taylor 展开：

$$
\begin{aligned}
I(x+u\Delta t,y+v\Delta t,t+\Delta t)
\approx {}&
I(x,y,t)\\
&+I_xu\Delta t\\
&+I_yv\Delta t\\
&+I_t\Delta t
\end{aligned}
$$

其中：

$$
I_x=\frac{\partial I}{\partial x},
\qquad
I_y=\frac{\partial I}{\partial y},
\qquad
I_t=\frac{\partial I}{\partial t}
$$

消去两侧相同的 $I(x,y,t)$，并除以 $\Delta t$，得到：

$$
\boxed{I_xu+I_yv+I_t=0}
$$

这就是 Optical Flow Constraint Equation（光流约束方程）。

它也可以写成向量形式：

$$
\begin{bmatrix}
I_x & I_y
\end{bmatrix}
\begin{bmatrix}
u\\
v
\end{bmatrix}
=-I_t
$$

**核心问题：**

每个像素只提供一个线性方程，但光流有两个未知量 $u,v$。

因此，仅靠单个像素的光流约束方程，通常无法唯一确定二维运动。

这一问题称为 Aperture Problem（孔径问题）。

#### 4. Aperture Problem：孔径问题

考虑一个只有单一方向边缘的图像区域。

在边缘处，图像梯度主要沿边缘的法线方向变化，而沿边缘方向的亮度变化可能很小。

假设：

$$
I_x=1,\qquad I_y=0,\qquad I_t=-2
$$

则光流约束方程为：

$$
u+0\cdot v-2=0
$$

所以：

$$
u=2
$$

但是，$v$ 可以取任意值。

因此，这个像素能够约束水平方向的运动，却无法确定垂直方向的运动。

**直观理解：**

- 只有水平边缘信息时，通常难以确定沿边缘方向的运动。
- 单一边缘主要提供垂直于边缘方向的运动信息。
- 具有多个不同方向纹理的区域，更容易约束完整的二维运动。

孔径问题说明：光流估计需要引入额外的信息，例如局部运动一致性、空间平滑性、图像金字塔或学习得到的特征。

#### 5. Lucas–Kanade Optical Flow

Lucas–Kanade（LK）是一种经典的局部光流估计方法。

其核心假设是：

**一个局部图像窗口内的像素具有近似相同的光流。**

设局部窗口中有 $N$ 个像素。对每个像素 $i$，都有一个光流约束方程：

$$
I_{x,i}u+I_{y,i}v=-I_{t,i}
$$

将所有像素的约束写成矩阵形式：

$$
A\mathbf{w}=\mathbf{b}
$$

其中：

$$
A=
\begin{bmatrix}
I_{x,1} & I_{y,1}\\
I_{x,2} & I_{y,2}\\
\vdots & \vdots\\
I_{x,N} & I_{y,N}
\end{bmatrix}
$$

$$
\mathbf{w}=
\begin{bmatrix}
u\\v
\end{bmatrix},
\qquad
\mathbf{b}=
-\begin{bmatrix}
I_{t,1}\\
I_{t,2}\\
\vdots\\
I_{t,N}
\end{bmatrix}
$$

当 $N>2$ 时，通常会得到一个超定方程组。由于图像梯度和亮度变化中存在噪声，所有方程不一定能被同时精确满足。

因此，LK 使用最小二乘法：

$$
\mathbf{w}^{*}
=
\arg\min_{\mathbf{w}}
\|A\mathbf{w}-\mathbf{b}\|_2^2
$$

对目标函数求导并令其为零：

$$
A^\top A\mathbf{w}=A^\top\mathbf{b}
$$

如果 $A^\top A$ 可逆，则：

$$
\boxed{
\mathbf{w}^{*}
=
(A^\top A)^{-1}A^\top\mathbf{b}
}
$$

在实际数值计算中，通常使用线性方程求解器或最小二乘求解器，而不是显式计算矩阵逆。

#### 6. Structure Tensor：结构张量

LK 方法中的关键矩阵是：

$$
G=A^\top A
$$

展开后：

$$
G=
\begin{bmatrix}
\sum_i I_{x,i}^2
&
\sum_i I_{x,i}I_{y,i}\\
\sum_i I_{x,i}I_{y,i}
&
\sum_i I_{y,i}^2
\end{bmatrix}
$$

该矩阵也称为结构张量（Structure Tensor），它反映局部区域中图像梯度的分布情况。

令其特征值为：

$$
\lambda_1,\lambda_2
$$

可以根据两个特征值判断局部区域是否适合估计光流。

**情况一：两个特征值都很大**

$$
\lambda_1\gg0,\qquad \lambda_2\gg0
$$

说明局部区域具有多个方向的明显梯度。

例如角点或纹理丰富的区域。

此时两个方向的运动通常都受到较强约束，光流估计相对可靠。

**情况二：一个特征值很大，另一个很小**

$$
\lambda_1\gg0,\qquad \lambda_2\approx0
$$

说明局部区域主要只有一个方向的梯度。

例如一条长直边缘。

此时沿某个方向的运动难以确定，容易出现孔径问题。

**情况三：两个特征值都很小**

$$
\lambda_1\approx0,\qquad \lambda_2\approx0
$$

说明局部区域缺乏明显纹理，例如一块近乎均匀的墙面。

此时图像本身几乎没有提供足够的运动信息。

**结论：**

LK 并不是在任何像素上都能稳定估计光流。局部纹理结构决定了运动估计问题是否具有足够的信息。

#### 7. Lucas–Kanade 的局限性

LK 的局部线性模型具有一定适用范围。

1. **小运动假设**

   一阶 Taylor 展开适合较小的位移。当两帧之间的运动过大时，线性近似可能不再准确。

2. **局部恒定运动假设**

   LK 假设一个窗口中的像素具有近似相同的运动。如果窗口跨越物体边界，或内部存在明显的运动差异，估计结果可能不准确。

3. **纹理不足**

   局部结构张量可能退化，使得两个运动分量无法可靠确定。

4. **亮度恒常假设**

   光照变化、遮挡和外观变化都会影响梯度约束的可靠性。

这些局限推动了多尺度光流、基于特征匹配的光流以及学习式光流的发展。

#### 8. Multi-Scale Optical Flow：多尺度光流

为了处理大位移，可以构建图像金字塔（Image Pyramid）。

设原始图像为：

$$
I^{(0)}=I
$$

逐渐下采样得到更粗糙的图像：

$$
I^{(1)},I^{(2)},\ldots,I^{(L)}
$$

其中，层级越高，图像分辨率越低。

假设物体在原始图像中移动 64 个像素。如果图像下采样 8 倍，那么同样的运动在该层中只对应：

$$
\frac{64}{8}=8
$$

个像素。

因此，在较粗的图像尺度上，大位移会变成较小的位移，更容易通过局部搜索或线性近似进行估计。

**Coarse-to-Fine 策略：**

1. 在较粗尺度上估计大致运动。
2. 将光流估计映射到更高分辨率。
3. 在更细尺度上进一步修正光流。
4. 重复该过程，直到达到所需的分辨率。

需要注意，恢复分辨率时不仅要放大光流场的空间尺寸，还要根据坐标尺度正确缩放位移数值。

多尺度方法可以扩大有效搜索范围，但也可能因为粗尺度丢失细节、错误匹配或遮挡而产生误差。

#### 9. 从传统方法走向学习式光流

传统方法通常需要人工设计：

- 用于匹配的图像表示或特征。
- 数据项，例如亮度一致性或特征相似性。
- 空间平滑性或局部运动一致性等约束。
- 多尺度搜索和迭代优化策略。

学习式光流则使用训练数据学习特征表示和运动更新规则。

其基本思路可以概括为：

1. 从两帧图像中提取特征。
2. 根据特征相似度寻找对应位置。
3. 根据匹配证据估计像素位移。
4. 迭代更新运动估计。
5. 通过监督学习优化预测结果。

RAFT（Recurrent All-Pairs Field Transforms）是具有代表性的学习式光流方法。

#### 10. RAFT：整体思路

RAFT 的主要组成部分包括：

**1. Feature Encoder**

对两帧图像提取特征：

$$
F_1=E(I_1),\qquad F_2=E(I_2)
$$

学习得到的特征比直接比较原始 RGB 亮度更灵活，可以帮助模型应对部分外观和光照变化。

**2. All-Pairs Correlation Volume**

计算第一帧每个特征位置与第二帧所有特征位置之间的相似度：

$$
C(p,q)=F_1(p)^\top F_2(q)
$$

其中：

- $p$：第一帧中的特征位置。
- $q$：第二帧中的候选匹配位置。

如果特征图有 $H'\times W'$ 个位置，那么共有：

$$
N=H'W'
$$

个位置，每个位置与另一帧的所有位置进行匹配，形成 $N\times N$ 的相关性矩阵。

**3. Correlation Pyramid**

对相关性体积的目标帧空间维度进行池化，构建多尺度相关性金字塔。

粗尺度可以扩大局部查询对应的有效搜索范围，细尺度则提供更精细的位置证据。

**4. Recurrent Update**

从初始光流开始，反复查询当前估计附近的相关性，并预测光流增量：

$$
f_{k+1}(p)=f_k(p)+\Delta f_k(p)
$$

更新网络使用循环隐藏状态保留和融合迭代信息。

**5. Upsampling**

最终将低分辨率光流恢复到目标输出分辨率。RAFT 使用学习得到的凸组合权重来组合邻域光流，而不是仅依赖固定的双线性插值权重。

#### 11. RAFT 与 Lucas–Kanade 的对比

| 方面 | Lucas–Kanade | RAFT |
|---|---|---|
| 匹配依据 | 原始图像梯度和亮度变化 | 学习得到的特征相似度 |
| 基本假设 | 局部窗口内运动近似恒定 | 通过学习的更新网络估计运动 |
| 搜索策略 | 局部线性求解，可结合金字塔 | 全对全相关性与多尺度局部查询 |
| 更新方式 | 求解最小二乘问题 | 迭代预测光流增量 |
| 先验与上下文 | 局部窗口假设等 | 学习得到的上下文特征与循环隐藏状态 |
| 训练需求 | 基本 LK 不需要训练 | 通常需要训练数据和监督损失 |

二者的目标相同：估计图像中的像素对应关系与二维运动。

它们的主要区别在于如何表示匹配信息，以及如何利用这些信息得到稳定的运动估计。

#### 12. 本节总结

本节的核心逻辑如下：

1. 光流描述相邻帧之间的二维像素位移。
2. 亮度恒常假设将运动估计转化为图像梯度约束。
3. 单个像素的约束方程不足以唯一确定两个运动分量，因此存在孔径问题。
4. Lucas–Kanade 利用局部运动一致性，通过最小二乘法求解光流。
5. 结构张量的特征值可以帮助判断局部运动估计是否可靠。
6. 图像金字塔可以缓解大位移带来的困难。
7. RAFT 通过学习式特征、全对全相关性、多尺度查询和循环更新来估计稠密光流。
8. 学习式方法并没有抛弃匹配与迭代优化的基本思想，而是学习其中的特征表示和更新规则。

### Lecture 1：表征学习与生成建模

#### Introduction to Representation Learning and Generative Modeling

**课程：** MIT 6.8300 — Advances in Computer Vision  
**对应日期：** 2026 年 3 月 31 日  
**主题：** 表征学习、概率密度估计、不确定性建模与生成模型

> 说明：本讲义依据课程大纲整理，并结合相关机器学习知识扩展，不是 MIT 官方课件或课堂逐字稿。

---

#### 0. 本讲的核心问题

计算机视觉模型究竟在学习什么？

假设输入一张图片 $x$，神经网络将它映射为一个特征向量：

$$
z = f_\theta(x), \qquad z \in \mathbb{R}^d
$$

其中：

- $x$：输入图像。
- $f_\theta$：参数为 $\theta$ 的神经网络编码器。
- $z$：图像的特征表示（representation）。
- $d$：特征向量的维度。

本讲主要回答以下问题：

1. 什么样的表征才算好？
2. 没有人工标签时，模型如何学到有意义的特征？
3. 表征学习与生成模型有什么联系？
4. 为什么压缩、去噪和缺失信息补全能够帮助模型学习？
5. CLIP、DINO 和扩散模型分别在学习什么？

**核心思想：** 表征学习寻找适合特定任务的信息编码方式；生成建模则从概率分布的角度描述数据及其不确定性。二者可以相互促进。

---

#### 1. 什么是好的表征？

##### 1.1 从原始图像到特征向量

一张 RGB 图片可以表示为：

$$
x \in \mathbb{R}^{H \times W \times 3}
$$

例如，一张 $224 \times 224$ 的图片包含：

$$
224 \times 224 \times 3 = 150528
$$

个像素值。

直接使用原始像素完成高层语义任务并不容易。两张语义上相同的猫的照片，可能因为位置、姿态、光照、背景和颜色不同，而具有完全不同的像素值。

因此，我们希望将图像转换为更有用的特征：

$$
x \xrightarrow{f_\theta} z
$$

理想情况下，$z$ 能够保留完成目标任务所需要的信息，同时减少无关变化的影响。

例如：

- **分类：** 保留与类别相关的信息。
- **检索：** 让语义相似的图片在特征空间中接近。
- **目标跟踪：** 保留物体身份和外观线索。
- **三维重建：** 保留几何结构、深度和视角相关的信息。
- **图文对齐：** 让图片与对应文字具有可比较的语义表示。

注意：**不存在对所有任务都最优的表征。** 对分类很好的特征，不一定适合精确定位、三维重建或图像生成。

##### 1.2 不变性与等变性

当输入发生变化时，表征应该如何变化？

###### 不变性（Invariance）

如果输入发生某种变化，但任务结果不应改变，那么我们希望表征尽可能保持不变：

$$
f(Tx) \approx f(x)
$$

其中 $T$ 表示某种输入变换。

例如，猫在图片中稍微向左移动，不应该让分类结果从“猫”变成“狗”。

###### 等变性（Equivariance）

如果输入发生变化，输出也应该按照相应规则变化，那么我们希望模型具有等变性：

$$
f(Tx) \approx T'f(x)
$$

其中：

- $T$：输入空间中的变换。
- $T'$：输出空间中对应的变换。

例如，图片中的物体向左移动，分割掩码也应相应向左移动。

两者的区别：

- 分类往往希望具有一定的不变性。
- 检测、分割和跟踪等空间任务通常需要保留几何对应关系。
- 三维视觉需要处理视角变化，同时保留重要的空间信息。

因此，不能简单地认为不变性越强越好。关键在于：**保留任务需要的信息，忽略任务不需要的变化。**

##### 1.3 如何评价表征？

可以从四个角度思考：

1. **任务性能：** 用于分类、检索、分割等任务时，效果如何？
2. **泛化能力：** 能否处理训练集中没有出现过的物体、背景和环境？
3. **鲁棒性：** 对光照、噪声、视角和裁剪等变化是否稳定？
4. **迁移能力：** 在一个数据集上学到的特征，能否迁移到其他任务或数据集？

一种常见的评估方法是冻结预训练编码器，只训练一个线性分类器：

$$
\hat{y} = Wz + b
$$

这称为 **Linear Probe（线性探测）**。

如果只使用一个线性层就能取得较好的效果，说明表征中可能已经包含容易被线性读出的任务相关信息。

但 Linear Probe 不能完整代表表征质量。它主要评估特定任务上的线性可分性，并不能直接衡量所有下游任务、鲁棒性或几何信息。

---

#### 2. 监督学习与自监督表征学习

##### 2.1 传统监督学习

假设训练数据为：

$$
\mathcal{D} = \{(x_i, y_i)\}_{i=1}^{N}
$$

其中 $x_i$ 是图像，$y_i$ 是人工标注的类别。

模型通常优化：

$$
\min_\theta
\frac{1}{N}
\sum_{i=1}^{N}
\mathcal{L}(f_\theta(x_i), y_i)
$$

这种方式直接告诉模型什么是正确答案。

优点是目标清晰；缺点是通常需要大量高质量标签，而且标签只覆盖预先定义的任务。

##### 2.2 自监督学习（Self-Supervised Learning）

自监督学习的基本思想是：不一定需要人工标注，可以从数据本身构造训练目标。

例如：

- 从同一张图片生成两个不同视图，让模型学习它们之间的关联。
- 随机遮住图像的一部分，让模型预测被遮挡的信息。
- 给图像加入噪声，让模型学习恢复原始信号。
- 利用视频相邻帧之间的时空关系构造学习目标。

一般形式可以写为：

$$
\min_\theta
\mathbb{E}_{x \sim p_{\text{data}}}
\left[
\mathcal{L}_{\text{pretext}}(x; \theta)
\right]
$$

其中 $\mathcal{L}_{\text{pretext}}$ 是代理任务的损失函数。

关键问题是：

**代理任务学到的能力，是否对真正关心的下游任务有帮助？**

如果代理目标与下游任务关系不大，模型可能学到一些容易利用、却没有多少实际价值的规律。

---

#### 3. 代理任务：压缩、去噪与缺失信息补全

课程大纲特别提到了三个例子：

- Compression：压缩。
- Denoising：去噪。
- Imputation：缺失信息补全。

它们展示了模型如何通过解决间接问题来学习数据结构。

##### 3.1 压缩（Compression）

压缩的目标是用更少的信息表示原始数据：

$$
z = f_\theta(x), \qquad
\hat{x} = g_\phi(z)
$$

其中：

- $z$：压缩后的表示。
- $\hat{x}$：重建结果。

最简单的重建损失是：

$$
\mathcal{L}_{\text{rec}} = \|x - \hat{x}\|_2^2
$$

但是，压缩并不是无损的。编码器必须决定哪些信息值得保留。

例如，一张图片里有一只猫和复杂的草地背景。若压缩得非常厉害，模型可能优先保留大致轮廓，却丢失细小纹理。

这里存在一个重要的权衡：

- 压缩率越高，能够保留的信息通常越少。
- 重建质量越高，往往需要更多的表示容量。
- 对不同任务来说，“重要信息”的定义并不相同。

在信息论中，可以把这个问题联系到率失真理论（Rate-Distortion Theory）：

$$
\min D(x, \hat{x})
\quad \text{subject to} \quad
R \leq R_{\max}
$$

其中：

- $D$：失真程度。
- $R$：编码所需的比特率。
- $R_{\max}$：允许的最大比特率。

**为什么压缩能够帮助表征学习？**

模型必须从大量像素中提取可复用的结构。但单纯的像素重建也可能使模型过分关注纹理或低层细节，不一定自动产生优秀的语义特征。

##### 3.2 去噪（Denoising）

设干净图像为 $x$，加入噪声后得到：

$$
\tilde{x} = x + \epsilon
$$

其中 $\epsilon$ 是噪声。

模型接收 $\tilde{x}$，试图预测原始图像：

$$
\hat{x} = g_\theta(\tilde{x})
$$

一种训练目标是：

$$
\mathcal{L}_{\text{denoise}}
=
\mathbb{E}_{x,\epsilon}
\left[
\|g_\theta(x+\epsilon)-x\|_2^2
\right]
$$

模型需要利用数据中稳定的统计规律来恢复合理的结果。

例如，图像中的噪点可能是随机的，但物体轮廓、边缘和局部结构通常更有规律。

不过，去噪不意味着模型必然学到高层语义。它可能主要学习低层图像统计规律，实际效果取决于噪声类型、网络结构、数据分布和训练目标。

##### 3.3 缺失信息补全（Imputation）

假设图像的一部分不可见。我们希望根据可见区域推断缺失区域。

设 $m$ 是可见区域掩码：

$$
x_{\text{visible}} = m \odot x
$$

其中 $\odot$ 表示逐元素相乘。

模型需要根据可见信息预测缺失内容：

$$
\hat{x}_{\text{missing}}
=
g_\theta(x_{\text{visible}}, m)
$$

例如，遮挡一张狗的照片中的一部分，让模型预测被遮挡的区域。

这个任务要求模型利用可见区域中的上下文信息，可能涉及物体形状、局部纹理、部件关系和场景结构。

但是，补全通常不是唯一答案。例如，只看到人的半张脸时，缺失部分可能存在多种合理结果。因此，我们还需要讨论概率分布和不确定性。

##### 3.4 三种代理任务的对比

| 代理任务 | 模型面对的问题 | 可能学到的规律 | 主要局限 |
|---|---|---|---|
| 压缩 | 如何用更少的信息表示数据？ | 低维结构、冗余关系 | 可能丢掉任务重要的信息 |
| 去噪 | 如何从受扰动的数据恢复信号？ | 稳定结构、数据统计规律 | 可能只学习低层特征 |
| 缺失补全 | 如何根据上下文推断未知部分？ | 局部结构、上下文关系 | 可能存在多个合理答案 |

这三个任务的共同点是：模型并不直接接收“猫”“汽车”等人工语义标签，却可以通过数据本身构造学习信号。

---

#### 4. 生成建模：从预测一个答案到描述可能的答案

##### 4.1 判别模型与生成模型

判别模型通常关注：

$$
p(y \mid x)
$$

即给定图像 $x$，预测标签 $y$ 的概率。

例如，分类器可能输出：

$$
p(y=\text{cat} \mid x) = 0.95
$$

生成模型则通常试图学习数据本身的分布：

$$
p_\theta(x)
$$

或者在给定条件 $c$ 时学习：

$$
p_\theta(x \mid c)
$$

例如，给定文字描述，生成符合描述的图像。

两类模型并不是绝对对立的。生成模型也可以用于判别相关任务，而判别模型也能学习丰富的特征表示。

##### 4.2 概率密度估计（Density Estimation）

现实中的图像来自某种复杂的数据分布 $p_{\text{data}}(x)$。我们通常不知道这个分布的精确形式，只能观察有限的数据样本：

$$
x_1, x_2, \ldots, x_N
\sim p_{\text{data}}(x)
$$

生成建模试图学习一个近似分布 $p_\theta(x)$，使它能够描述真实数据的统计规律。

一种经典目标是最大化对数似然：

$$
\max_\theta
\sum_{i=1}^{N}
\log p_\theta(x_i)
$$

直觉是：模型应该给训练数据中的真实样本分配较高的概率密度。

但这不等于模型一定能生成高质量图片。模型的表示能力、优化过程、数据覆盖范围以及生成算法都会影响最终结果。

还要区分两个概念：

- **概率密度高：** 模型认为该位置附近的数据较为常见。
- **生成质量高：** 生成样本在视觉、语义或应用指标上表现良好。

二者相关，但并不完全等价。

##### 4.3 不确定性建模（Uncertainty Modeling）

如果输入信息不完整，模型可能无法确定唯一答案。

例如，只观察到一只动物的尾巴和部分身体，可能无法判断它究竟是哪种动物。一个合理的模型应该允许多个可能结果，而不是假装自己百分之百确定。

在条件生成中，我们可以写为：

$$
p_\theta
\left(
x_{\text{missing}}
\mid x_{\text{visible}}
\right)
$$

这个分布描述了：给定可见区域后，缺失区域有哪些可能的取值，以及它们相对有多合理。

从分布中采样：

$$
x_{\text{missing}}
\sim
p_\theta
\left(
x_{\text{missing}}
\mid x_{\text{visible}}
\right)
$$

便可能得到多个不同但合理的补全结果。

这也解释了为什么单一的均方误差重建可能有局限：当真实答案存在多个可能性时，直接回归一个平均结果，可能产生模糊或不够自然的图像。

**重要启示：** 对于存在歧义的问题，预测一个答案和描述答案的不确定性，是两种不同的建模目标。

---

#### 5. 表征学习与生成建模有什么关系？

二者关注的问题不同，但可以互相连接。

表征学习希望找到有用的编码：

$$
z = f_\theta(x)
$$

生成建模希望描述数据的分布：

$$
p_\theta(x)
$$

##### 5.1 在潜在空间中生成

先把图像编码成潜在变量 $z$，再从潜在变量重建图像：

$$
x \xrightarrow{f_\theta} z
\xrightarrow{g_\phi} \hat{x}
$$

如果潜在变量具有合适的结构，生成模型就可以尝试在潜在空间中学习或采样，再将结果解码回图像空间。

这类思路出现在自编码器、变分自编码器（VAE）以及潜在扩散模型等方法中，但不同方法对潜在变量和概率分布的定义并不相同。

##### 5.2 通过生成任务学习表征

如果模型需要补全被遮挡的图像区域，就必须利用可见内容中的结构信息。

如果模型需要从噪声中恢复图像，就需要学习数据在不同噪声程度下的规律。

这些目标可能帮助模型学习可复用的特征。但生成效果好并不自动意味着表征适合分类、检索或其他任务；仍然需要针对目标任务进行评估。

##### 5.3 统一视角

1. 数据中存在大量规律与冗余。
2. 模型通过某种训练目标学习这些规律。
3. 学到的内部表示可以用于预测、检索、重建或生成。
4. 训练目标决定模型更倾向保留哪些信息。
5. 最终表征是否有用，取决于它与目标任务之间的关系。

---

#### 6. 联系到 CLIP、DINO 和扩散模型

##### 6.1 CLIP：通过跨模态对比学习建立语义空间

CLIP 使用图像编码器和文本编码器：

$$
z_I = f_\theta(x), \qquad
z_T = g_\phi(t)
$$

通常将特征归一化，再计算图像与文本的余弦相似度：

$$
s(x,t)=
\frac{z_I^\top z_T}
{\|z_I\|_2 \|z_T\|_2}
$$

训练目标会鼓励匹配的图文对具有较高相似度，同时区分不匹配的样本。

CLIP 的重点不只是“识别图片是什么”，而是学习一种能够连接视觉和语言的表征空间。

从本讲角度看：

- **表征是什么？** 图像与文本的向量表示。
- **学习信号是什么？** 图文配对关系及对比目标。
- **表征希望保留什么？** 有助于跨模态语义匹配的信息。
- **可能的局限是什么？** 训练数据和目标会影响模型对细粒度空间信息、长尾概念和分布外样本的表现。

##### 6.2 DINO：从图像自身学习视觉特征

DINO 系列方法属于自监督视觉表征学习的重要方向。

其核心思想是利用不同图像视图之间的关系构造训练信号，而不是依赖每张图像的人工类别标签。

从概念上看：

1. 从同一张图像生成不同视图，例如裁剪或改变尺度。
2. 将视图输入网络。
3. 通过训练目标，使不同视图的输出具有期望的关系。
4. 让网络逐渐学习可迁移的视觉结构。

具体实现会因 DINO 版本而异，不能将所有版本简单归结为同一个损失函数。

DINO 回答了一个关键问题：

**没有人工类别标签，模型能否仅凭数据内部的结构学到有用的视觉特征？**

评估这类表征时，可以考察线性探测、迁移任务，以及特征对物体区域或语义结构的反映。

##### 6.3 扩散模型：通过逐步去噪学习数据分布

扩散模型通常构造一个逐渐向数据加入噪声的过程，再训练模型学习逆向去噪。

一个常见的前向过程写为：

$$
x_t =
\sqrt{\bar{\alpha}_t}x_0
+
\sqrt{1-\bar{\alpha}_t}\epsilon,
\qquad
\epsilon \sim \mathcal{N}(0,I)
$$

其中：

- $x_0$：原始数据。
- $x_t$：第 $t$ 个噪声阶段的数据。
- $\bar{\alpha}_t$：控制原始信号保留比例的系数。
- $\epsilon$：高斯噪声。

常见训练目标是预测噪声：

$$
\mathcal{L}_{\text{diff}}
=
\mathbb{E}_{x_0,t,\epsilon}
\left[
\|\epsilon-\epsilon_\theta(x_t,t)\|_2^2
\right]
$$

训练完成后，模型可以通过逐步去噪生成样本。不同扩散模型可能预测噪声、原始样本、速度参数或其他等价参数化量。

从本讲角度看：

- 去噪是构造学习信号的一种方式。
- 模型通过训练学习数据在不同噪声程度下的统计结构。
- 生成过程体现了模型对数据分布的近似。
- 对于条件生成，模型还需要利用文本、类别或其他条件信息。

需要注意：扩散模型的去噪目标与传统去噪自编码器有联系，但它们的训练过程、噪声设计和生成机制并不完全相同。

---

#### 7. 本讲的概念地图

可以用下面这条主线串起整个讲座：

**数据 $x$**

↓ 通过编码器、代理任务或概率模型进行学习

**表征 $z$ 与数据分布 $p_\theta(x)$**

↓ 用不同目标约束模型学习什么

- **压缩：** 保留有限容量下的重要信息。
- **去噪：** 从扰动中恢复数据规律。
- **补全：** 根据上下文推断缺失信息。
- **对比学习：** 让相关样本在表征空间中具有期望的关系。
- **生成建模：** 描述数据及其可能性。

↓ 最终服务于

**分类、检索、分割、跨模态理解、图像生成、视频生成等任务。**

模型不是凭空“理解”数据，而是在训练目标、数据分布和模型结构共同作用下，学习某种有用的统计规律。

---

#### 8. 建议重点掌握的概念

| 概念 | 需要能回答的问题 |
|---|---|
| Representation | 为什么需要把原始图像编码成特征？ |
| Invariance | 哪些输入变化不应该影响任务结果？ |
| Equivariance | 哪些输入变化应该在输出中体现出来？ |
| Self-supervised learning | 没有人工标签时，学习信号从哪里来？ |
| Surrogate task | 为什么代理任务可能帮助下游任务？ |
| Density estimation | 模型如何学习数据的概率分布？ |
| Uncertainty modeling | 为什么一个问题可能存在多个合理答案？ |
| Generative modeling | 生成模型与判别模型的主要目标有什么区别？ |
| Linear probe | 如何初步评价预训练表征？ |

---

#### 9. 自测题

建议先独立回答，再查看参考答案。

##### 问题 1

为什么一个在 ImageNet 分类任务上表现很好的表征，不一定适合三维重建？

##### 问题 2

不变性和等变性的区别是什么？请各举一个计算机视觉的例子。

##### 问题 3

为什么只使用像素级重建损失，不一定能得到最适合语义分类的特征？

##### 问题 4

生成模型学习 $p(x)$ 与分类模型学习 $p(y \mid x)$ 有什么区别？

##### 问题 5

为什么图像补全可能需要输出一个分布，而不只是一个确定的图像？

##### 问题 6

CLIP、DINO 和扩散模型分别主要通过什么训练信号学习？

##### 参考答案

**1.** 分类更关注区分类别所需的信息；三维重建则需要保留几何、深度、视角等信息。分类表征可能压缩或忽略部分几何细节。

**2.** 不变性意味着某种输入变换不应改变表征，例如轻微平移不应改变物体类别；等变性意味着输出应随输入按对应规则变化，例如图像平移后，分割掩码也应平移。

**3.** 像素级损失强调重建细节，而语义分类强调类别相关信息。低层纹理与像素误差并不总是与高层语义一致。

**4.** $p(x)$ 描述数据本身的分布；$p(y \mid x)$ 描述给定输入后标签的条件分布。前者更直接对应生成建模，后者更直接对应判别预测。

**5.** 同一组可见信息可能对应多个合理的缺失区域。概率分布可以表达这些可能性，而单一预测只能给出一个结果。

**6.** CLIP 主要利用图文匹配关系进行对比学习；DINO 利用不同图像视图之间的自监督信号学习特征；扩散模型通常通过不同噪声阶段的去噪目标学习数据分布。

---

#### 10. 推荐学习顺序

1. **先理解第 1 节：** 好的表征到底意味着什么。
2. **再看第 2、3 节：** 没有人工标签时，模型如何构造学习目标。
3. **然后看第 4、5 节：** 从确定性预测过渡到概率分布和生成建模。
4. **最后看第 6 节：** 把抽象概念对应到 CLIP、DINO 和扩散模型。

完成后，试着用自己的话回答三个核心问题：

- 模型究竟在学习什么？
- 训练目标如何影响它学到的表示？
- 如何判断这些表示对真正关心的任务有用？

如果这三个问题能够讲清楚，这一讲的核心思想就基本掌握了。

**延伸阅读：** 接下来可以学习课程中的 Diffusion 1，进一步理解去噪目标、概率建模以及生成过程之间的联系；也可以学习 Non-Generative Representation Learning，比较自监督表征学习与生成式学习的异同。

### Diffusion 1：Flow and Diffusion Models

**课程：** MIT 6.8300 — Advances in Computer Vision  
**核心主题：** 生成建模、ODE/SDE、Flow Models、Flow Matching

#### 1. 从生成问题到概率分布

##### 1.1 如何定义“生成得好”？

假设我们希望模型生成一张符合提示词 `A picture of a dog` 的图片。

一张图片生成得好不好，最初看起来是主观问题。课程引入了一个概率视角：

> 一张图片越符合真实数据的统计规律，它在目标数据分布下出现的可能性就越高。

令 $x$ 表示一张图片，$p_{\text{data}}(x)$ 表示目标数据分布的概率密度。

生成建模的目标可以写成：

$$
x \sim p_{\text{data}}(x)
$$

这里的 $p_{\text{data}}$ 描述我们希望生成的数据具有怎样的分布。对于图像生成任务，它可以理解为真实图像在图像空间中的分布。

需要注意：

- $p_{\text{data}}(x)$ 通常是未知的。
- 我们拥有的是从这个分布采样得到的训练数据，而不是完整的概率密度函数。
- 模型的目标是学习生成符合该分布的新样本，而不是简单地复制训练图片。

对于文本条件生成，更完整的目标是从条件分布中采样：

$$
x \sim p_{\text{data}}(x \mid y)
$$

其中 $y$ 表示文本提示词或其他条件。

##### 1.2 生成模型的基本思想

直接从复杂的数据分布中采样很困难。因此，生成模型通常从一个容易采样的初始分布出发，再将样本转换成目标数据。

$$
z \sim p_0(z)
\quad \xrightarrow{\text{Generative Model}} \quad
x \sim p_{\text{data}}(x)
$$

常见的初始分布是标准高斯分布：

$$
z \sim \mathcal{N}(0,I)
$$

**直觉：** 初始样本可以看作随机噪声。模型逐步改变这个样本，使最终结果符合目标数据的统计规律。

在 Flow 和 Diffusion 模型中，这种转换可以通过连续时间的动力系统来描述。

#### 2. Flow 与 Diffusion：用微分方程描述生成过程

##### 2.1 Flow Model：通过 ODE 生成样本

Flow Model 使用常微分方程（Ordinary Differential Equation，ODE）描述样本随时间的变化：

$$
\frac{dX_t}{dt} = u_t^\theta(X_t)
$$

其中：

- $X_t$：时间 $t$ 的样本状态。
- $t$：连续时间，课程采用从 $t=0$ 到 $t=1$ 的生成过程。
- $u_t^\theta(x)$：神经网络表示的速度场（vector field）。
- $\theta$：神经网络的可学习参数。

速度场告诉我们：当样本位于 $x$、时间为 $t$ 时，下一瞬间应该朝哪个方向移动，以及移动多快。

生成过程为：

1. 从初始分布采样 $X_0 \sim p_0$。
2. 使用神经网络计算当前速度 $u_t^\theta(X_t)$。
3. 根据 ODE 数值积分，逐步更新样本。
4. 得到终点 $X_1$，作为生成结果。

##### 2.2 Euler 方法：如何数值求解 ODE？

实际计算机无法直接执行连续的无限小变化，因此需要使用数值积分方法。

最简单的方法是 Euler 方法：

$$
X_{t+\Delta t}
\approx X_t + \Delta t\,u_t^\theta(X_t)
$$

其中 $\Delta t$ 是时间步长。

**理解公式：**

- $u_t^\theta(X_t)$ 给出当前移动方向和速度。
- $\Delta t\,u_t^\theta(X_t)$ 是这一小段时间内的近似位移。
- 不断重复这个过程，就可以得到从初始噪声到生成样本的轨迹。

若将时间区间 $[0,1]$ 划分为 $N$ 个等长步骤，则：

$$
\Delta t = \frac{1}{N}
$$

$$
X_{k+1}
= X_k + \frac{1}{N}u_{t_k}^\theta(X_k)
$$

这里的 $k$ 是离散计算步骤的编号，不应与连续时间 $t$ 混淆。

一般而言，步长越小，Euler 方法对连续轨迹的近似通常越精细，但计算成本也越高。

##### 2.3 Diffusion Model：通过 SDE 生成样本

Diffusion 模型可以使用随机微分方程（Stochastic Differential Equation，SDE）描述：

$$
dX_t = u_t^\theta(X_t)\,dt + \sigma_t\,dW_t
$$

其中：

- $u_t^\theta(X_t)\,dt$：漂移项，描述有方向的变化。
- $dW_t$：布朗运动的随机增量。
- $\sigma_t$：扩散系数，控制随机扰动的强度。

与 ODE 相比，SDE 在确定性变化之外加入了随机噪声。

**直觉对比：**

- ODE：给定初始状态和速度场，轨迹由动力学确定。
- SDE：除了漂移变化，还存在随机扰动；即使漂移项相同，不同的随机过程也可能产生不同轨迹。

这并不意味着 ODE 只能用于 Flow、SDE 只能用于 Diffusion 的所有实现。它们是本讲介绍的两种核心生成描述方式，后续课程会进一步讨论二者之间的联系。

##### 2.4 ODE 与 SDE 对比

| 对比项 | Flow：ODE | Diffusion：SDE |
|---|---|---|
| 基本方程 | $\frac{dX_t}{dt}=u_t^\theta(X_t)$ | $dX_t=u_t^\theta(X_t)dt+\sigma_t dW_t$ |
| 状态变化 | 由速度场确定 | 由漂移项和随机扰动共同决定 |
| 随机性 | 给定初始状态后，轨迹是确定的 | 演化过程中包含随机性 |
| 数值模拟 | 例如 Euler 积分 | 需要同时模拟漂移与随机增量 |
| 共同目标 | 将初始分布转换成目标数据分布 | 将初始分布转换成目标数据分布 |

##### 2.5 本讲的时间约定

本讲采用以下生成方向：

$$
t=0:\text{Noise}
\quad \longrightarrow \quad
t=1:\text{Data}
$$

也就是说：

- $X_0$：从初始噪声分布采样。
- $X_1$：生成的最终样本。
- $X_t$：生成过程中的中间状态。

**注意：** 不同 Diffusion 文献可能采用不同的时间方向。阅读公式时，应先确认作者如何定义 $t=0$ 和 $t=1$。

#### 3. Flow Matching：如何训练速度场？

##### 3.1 训练目标

Flow Model 的采样过程依赖速度场 $u_t^\theta(x)$，因此训练的关键是让神经网络学会正确的速度场。

但真实数据分布的完整演化规律通常未知，我们也无法直接获得真实的边缘速度场。因此，课程采用 Flow Matching 思路：构造从噪声到数据的概率路径，并利用可计算的监督信号训练神经网络。

##### 3.2 Conditional 与 Marginal

这是 Flow Matching 中非常重要的一组术语。

| 术语 | 含义 | 直观理解 |
|---|---|---|
| Conditional（条件） | 针对一个给定的数据点 | 固定一个样本，研究与它关联的路径 |
| Marginal（边缘） | 跨越整个数据分布 | 考虑不同数据样本共同形成的整体分布 |

设 $z$ 是从真实数据分布采样得到的一个数据点。

对于固定的 $z$，我们可以定义条件概率路径：

$$
p_t(x \mid z)
$$

它描述在给定数据点 $z$ 的情况下，时间 $t$ 的样本状态如何分布。

对所有数据点进行平均，就得到边缘概率路径：

$$
p_t(x)
=
\int p_t(x \mid z)\,p_{\text{data}}(z)\,dz
$$

这条边缘路径描述整个生成过程中，所有样本的总体分布如何随时间变化。

##### 3.3 从条件路径到边缘路径

课程强调：我们希望模型学习的是能够驱动整体分布演化的边缘速度场。

记：

- $u_t(x \mid z)$：条件速度场。
- $u_t(x)$：边缘速度场。

二者的任务不同：

- 条件速度场描述给定数据点 $z$ 时，如何沿对应的条件概率路径运动。
- 边缘速度场描述整体概率分布应如何运动，才能沿目标边缘概率路径演化。

条件路径通常更容易构造，条件速度场也更容易计算。因此，可以使用大量不同数据点对应的条件监督信号来训练神经网络，使其近似边缘速度场。

**关键理解：** 条件速度场与边缘速度场不能简单地视为处处相同。Flow Matching 的核心，是利用可计算的条件目标来学习所需的边缘动力学。

##### 3.4 连续性方程：速度场如何改变概率分布？

为了理解速度场与概率分布之间的关系，需要引入连续性方程（Continuity Equation）：

$$
\frac{\partial p_t(x)}{\partial t}
=
-\nabla \cdot \left(p_t(x)u_t(x)\right)
$$

其中：

- $p_t(x)$：时间 $t$ 的概率密度。
- $u_t(x)$：对应的速度场。
- $\nabla\cdot$：散度运算，用于描述向量场在某一点附近的净流出程度。

**直觉：概率质量守恒。**

可以把概率密度想象成流体的密度，把速度场想象成流体的流速：

- 如果某个区域流出的概率质量多于流入的概率质量，该区域的概率密度就会下降。
- 如果流入多于流出，该区域的概率密度就会上升。

连续性方程刻画的正是这种概率质量随时间的变化规律。

它建立了两个关键对象之间的联系：

$$
\boxed{
\text{速度场}
\quad \longleftrightarrow \quad
\text{概率分布的演化}
}
$$

如果速度场设计得当，沿着该速度场求解 ODE，就能够使样本的边缘分布按照预期的概率路径演化。

##### 3.5 Flow Matching 的训练逻辑

可以把本讲的 Flow Matching 思路整理成以下流程：

1. **采样数据点：** 从真实数据分布采样 $z$。
2. **构造概率路径：** 定义噪声与数据之间的条件概率路径。
3. **获得监督信号：** 计算该条件路径对应的目标速度。
4. **训练神经网络：** 使 $u_t^\theta(x)$ 逼近目标速度。
5. **执行生成：** 从初始噪声出发，使用训练好的边缘速度场求解 ODE。
6. **返回终点：** 得到生成样本 $X_1$。

课程随后以 Meta MovieGen 和 Stable Diffusion 3 为例，说明 Flow Matching 可以用于大规模视频与图像生成。

##### 3.6 为什么不直接用条件速度场生成？

这里容易产生一个误解：既然条件速度场更容易计算，为什么还要学习边缘速度场？

因为生成时通常只有一个随机初始噪声样本，并不知道它应该对应哪个真实数据点 $z$。

因此，模型不能依赖一个预先指定的真实数据点来决定整个生成过程。它需要学习适用于整体分布的速度场，从噪声出发生成合理的数据样本。

条件路径和条件速度场主要提供训练监督；边缘速度场则承担实际生成时的整体分布演化任务。

#### 4. Diffusion 1 总结

##### 4.1 核心知识地图

```text
生成问题
  ↓
将生成理解为从数据分布中采样
  ↓
从容易采样的初始分布出发
  ↓
用连续时间动力系统完成分布转换
  ├── ODE → Flow Model
  └── SDE → Diffusion Model
  ↓
Flow Model 的训练
  ├── 构造条件概率路径
  ├── 获得条件速度监督
  ├── 学习边缘速度场
  └── 使用 ODE 数值积分生成样本
```

##### 4.2 必须掌握的公式

**（1）Flow 的 ODE**

$$
\frac{dX_t}{dt}=u_t^\theta(X_t)
$$

含义：神经网络给出当前样本应当如何移动。

**（2）Euler 数值积分**

$$
X_{t+\Delta t}\approx X_t+\Delta t\,u_t^\theta(X_t)
$$

含义：用一系列小步近似连续生成轨迹。

**（3）Diffusion 的 SDE**

$$
dX_t=u_t^\theta(X_t)\,dt+\sigma_t\,dW_t
$$

含义：漂移项与随机噪声共同决定状态变化。

**（4）连续性方程**

$$
\frac{\partial p_t(x)}{\partial t}
=-\nabla\cdot(p_t(x)u_t(x))
$$

含义：速度场决定概率质量如何在空间中流动。

##### 4.3 复习时要能回答的问题

- 为什么生成建模可以理解为从目标数据分布中采样？
- 为什么模型可以从简单的高斯噪声分布出发？
- ODE 与 SDE 在生成过程中的区别是什么？
- Euler 方法如何近似求解 ODE？
- Conditional 与 Marginal 分别指什么？
- 连续性方程如何将速度场与概率分布的演化联系起来？
- 为什么 Flow Matching 可以利用条件速度来学习边缘速度场？
- 为什么训练时使用条件监督，不代表生成时必须知道对应的真实数据点？

##### 4.4 一句话记忆

**Diffusion 1 的主线：** 将生成看成从噪声分布到数据分布的转换；用 ODE 或 SDE 描述转换过程；对于 Flow Model，则通过构造条件概率路径和条件速度监督，训练能够驱动整体分布演化的速度场。

### D2. Flow and Diffusion Models — 2

#### 本讲学习路线

本讲建立在 D1 的 Flow Matching 基础上，主要回答五个问题：

1. **Score Matching：** 除了学习速度场，为什么还可以学习概率密度的梯度？
2. **Diffusion Sampling：** 如何利用 score function 实现随机采样？
3. **Classifier-Free Guidance（CFG）：** 如何让生成结果更符合文本提示？
4. **Neural Network Architectures：** 为什么生成模型需要 U-Net 或 Diffusion Transformer（DiT）？
5. **Large-scale Models：** Stable Diffusion 3 和 MovieGen 如何应用这些思想？

建议重点掌握前三部分的数学推导，后两部分重点理解设计动机与模型之间的联系。

---

### 1. 从 Flow Matching 到 Score Matching

#### 1.1 回顾：Flow Matching 学习什么？

在 D1 中，我们将生成过程表示为一个随时间变化的速度场：

$$
\frac{dX_t}{dt}=u_t^\theta(X_t)
$$

其中：

- $X_t$：时刻 $t$ 的样本，例如一张正在生成的图像。
- $u_t^\theta(x)$：神经网络预测的速度场。
- $\theta$：神经网络参数。

给定初始噪声 $X_0$，通过求解 ODE，逐步将样本从噪声分布移动到目标数据分布。

Flow Matching 的训练目标可以概括为：让神经网络预测的速度尽可能接近目标速度。

$$
\mathcal{L}_{\mathrm{FM}}
=
\mathbb{E}\left[
\left\|u_t^\theta(X_t)-u_t^*(X_t)\right\|^2
\right]
$$

这里的 $u_t^*$ 表示对应训练构造下的目标速度。实际训练中，我们可以利用条件概率路径构造容易计算的监督信号。

**核心问题：** 速度场不是描述概率分布变化的唯一方式。我们还可以学习一个与概率密度相关的量：Score Function。

#### 1.2 Score Function 的定义

Score Function 定义为对数概率密度关于输入的梯度：

$$
s_t(x)=\nabla_x\log p_t(x)
$$

其中：

- $p_t(x)$：时刻 $t$ 的概率密度。
- $\log p_t(x)$：对数概率密度。
- $\nabla_x$：对输入 $x$ 求梯度。

直觉上，score 描述了概率密度随空间位置变化的方向与强度。

因为

$$
\nabla_x\log p_t(x)
=
\frac{\nabla_x p_t(x)}{p_t(x)}
$$

所以 score 反映的是概率密度的相对变化率，而不只是密度本身的大小。

**注意：** Score 不等于概率密度，也不直接等于速度。它们是不同的数学对象，但在特定的概率路径下，可以建立明确的转换关系。

#### 1.3 一个简单例子：标准高斯分布

设

$$
p(x)=\mathcal{N}(x;0,I)
$$

忽略与 $x$ 无关的常数，其对数概率密度为

$$
\log p(x)=-\frac{1}{2}x^\top x+C
$$

对 $x$ 求梯度：

$$
\nabla_x\log p(x)=-x
$$

因此：

$$
\boxed{s(x)=-x}
$$

这意味着 score 指向高斯分布的高密度区域：当 $x$ 偏离原点时，score 指向原点方向。

对于一般的概率分布，score 通常更加复杂，需要通过神经网络进行学习。

#### 1.4 Conditional 与 Marginal Score

本讲延续 D1 中 Conditional 和 Marginal 的区分。

**Conditional Score：**

$$
s_t(x\mid z)
=
\nabla_x\log p_t(x\mid z)
$$

它描述给定数据样本 $z$ 后，条件概率路径的 score。

**Marginal Score：**

$$
s_t(x)=\nabla_x\log p_t(x)
$$

它描述将所有可能的条件样本综合起来后，边缘概率路径的 score。

二者的区别在于是否显式给定某个数据样本 $z$。

和 Flow Matching 类似，我们希望通过容易计算的条件目标，学习真正需要的边缘 score。

---

### 2. Gaussian Probability Paths：速度场与 Score 的联系

#### 2.1 CondOT 线性概率路径

D1 和本讲都使用一个重要的例子：Conditional Optimal Transport（CondOT）线性路径。

设：

- $\epsilon\sim\mathcal{N}(0,I)$：初始噪声。
- $z\sim p_{\mathrm{data}}$：目标数据样本。

构造路径：

$$
x_t=(1-t)\epsilon+tz
$$

当 $t=0$ 时：

$$
x_0=\epsilon
$$

当 $t=1$ 时：

$$
x_1=z
$$

所以这条路径将噪声与数据通过线性插值连接起来。

对 $t$ 求导：

$$
\frac{dx_t}{dt}=z-\epsilon
$$

因此，沿着这条条件路径的目标速度为

$$
\boxed{u_t(x_t\mid z)=z-\epsilon}
$$

这里有一个很重要的观察：**条件速度是线性路径的导数，因此计算起来非常简单。**

#### 2.2 计算 Conditional Score

给定目标样本 $z$，有

$$
x_t=(1-t)\epsilon+tz
$$

将其改写为

$$
x_t\mid z
\sim
\mathcal{N}\left(tz,(1-t)^2I\right)
$$

对于均值为 $\mu$、协方差为 $\sigma^2 I$ 的高斯分布：

$$
\nabla_x\log p(x)
=
-\frac{x-\mu}{\sigma^2}
$$

因此，这条条件概率路径的 score 为

$$
s_t(x_t\mid z)
=
-\frac{x_t-tz}{(1-t)^2}
$$

由于

$$
x_t-tz=(1-t)\epsilon
$$

代入后得到

$$
\boxed{
s_t(x_t\mid z)
=
-\frac{\epsilon}{1-t}
}
\qquad t<1
$$

这个公式是本讲的重要结果。

它表明：在高斯概率路径下，Conditional Score 和 Conditional Vector Field 都可以用简单的线性表达式计算。

需要注意，$t=1$ 时公式的分母为零，因此该表达式不能直接在端点 $t=1$ 使用。

#### 2.3 为什么早期 Diffusion Models 学习 Score？

对于 CondOT 路径，我们已经得到：

$$
u_t(x_t\mid z)=z-\epsilon
$$

以及

$$
s_t(x_t\mid z)=-\frac{\epsilon}{1-t}
$$

因此，可以从 score 中恢复噪声：

$$
\epsilon=-(1-t)s_t(x_t\mid z)
$$

再代入速度公式：

$$
u_t(x_t\mid z)
=
z+(1-t)s_t(x_t\mid z)
$$

这说明，在该条件高斯路径下，速度与 score 存在明确的转换关系。

不过，上式使用的是条件 score。实际生成时，我们通常关心的是边缘速度场和边缘 score。二者之间的转换需要使用对应的边缘概率路径及其参数化公式，不能简单地将所有条件量直接替换成边缘量。

**本讲的核心结论：** 对适当的高斯概率路径，学习边缘 Vector Field 与学习边缘 Score 可以相互转换。因此，Flow Matching 和 Score-based Diffusion 虽然采用不同的训练目标，但在特定条件下具有紧密的数学联系。

---

### 3. Denoising Score Matching（DSM）

#### 3.1 为什么不直接学习边缘 Score？

目标是学习

$$
s_t(x)=\nabla_x\log p_t(x)
$$

但是，真实边缘概率密度 $p_t(x)$ 通常很难直接计算，因此也难以直接计算它的 score。

一种解决办法是：利用已知的条件概率路径，训练模型预测 Conditional Score，再通过条件样本的训练目标学习 Marginal Score。

这就是 Denoising Score Matching（DSM）的基本思路。

#### 3.2 将 Score Matching 转换成噪声预测

对于线性高斯路径：

$$
x_t=tz+(1-t)\epsilon
$$

其中：

$$
\epsilon\sim\mathcal{N}(0,I)
$$

我们已经推导出：

$$
s_t(x_t\mid z)=-\frac{\epsilon}{1-t}
$$

因此，如果神经网络预测噪声 $\epsilon$，就可以构造 score 的预测值：

$$
s_t^\theta(x_t)
=
-\frac{\epsilon_\theta(x_t,t)}{1-t}
$$

这里的 $\epsilon_\theta$ 表示网络预测的噪声。

这解释了为什么一些 Diffusion Models 的训练可以被理解为噪声预测：网络需要估计被加入样本中的噪声，从而恢复对应的 score 信息。

#### 3.3 DSM 的训练目标

一种概念性的条件 score matching 目标为

$$
\mathcal{L}_{\mathrm{DSM}}
=
\mathbb{E}_{t,z,x_t}
\left[
\left\|
s_t^\theta(x_t)
-
s_t(x_t\mid z)
\right\|^2
\right]
$$

这里：

- $z$ 是从数据分布中采样的数据。
- $x_t$ 是从对应条件概率路径采样得到的扰动样本。
- $s_t(x_t\mid z)$ 是可计算的条件 score。
- $s_t^\theta(x_t)$ 是模型预测的 score。

对于前面给出的线性高斯路径，将目标 score 代入：

$$
\mathcal{L}_{\mathrm{DSM}}
=
\mathbb{E}
\left[
\left\|
s_t^\theta(x_t)+\frac{\epsilon}{1-t}
\right\|^2
\right]
$$

不同实现可以采用不同的权重或等价的预测参数化；这里的表达式用于展示基本原理。

**需要记住的逻辑：**

1. 对数据加入噪声，构造 $x_t$。
2. 根据已知的条件分布，计算条件 score。
3. 训练神经网络预测该 score，或者预测与之等价的噪声。
4. 通过这种训练方式学习生成所需的边缘 score。

#### 3.4 数值稳定性

由于

$$
s_t(x_t\mid z)=-\frac{\epsilon}{1-t}
$$

当 $t$ 接近 $1$ 时，分母变小，score 的数值可能变大。

这会带来数值稳定性方面的挑战。因此，训练时的时间采样、损失加权以及预测参数化都可能影响实际训练效果。

**学习重点：** 理解为什么预测噪声能够学习 score，比死记某个具体损失函数更重要。

---

### 4. Fokker–Planck Equation：SDE 如何改变概率分布？

#### 4.1 从 ODE 走向 SDE

D1 中，Flow 使用 ODE：

$$
dX_t=u_t(X_t)\,dt
$$

Diffusion 则可以使用 SDE：

$$
dX_t=u_t(X_t)\,dt+\sigma_t\,dW_t
$$

其中：

- $u_t(X_t)\,dt$：漂移项，决定平均运动趋势。
- $\sigma_t$：噪声强度。
- $dW_t$：布朗运动增量。

ODE 只包含确定性的动力学；SDE 额外加入随机扰动。

但这引出了一个问题：**如果单个样本在随机运动，那么整个样本群体的概率密度会怎样变化？**

Fokker–Planck Equation 正是描述这个问题的方程。

#### 4.2 Fokker–Planck 方程

对于以下 SDE：

$$
dX_t=u_t(X_t)\,dt+\sigma_t\,dW_t
$$

在标准假设下，其概率密度 $p_t(x)$ 满足

$$
\boxed{
\frac{\partial p_t(x)}{\partial t}
=
-\nabla\cdot\left(p_t(x)u_t(x)\right)
+
\frac{\sigma_t^2}{2}\Delta p_t(x)
}
$$

其中：

- $\frac{\partial p_t}{\partial t}$：某个位置处的概率密度随时间的变化。
- $-\nabla\cdot(p_tu_t)$：由漂移速度造成的概率质量输运。
- $\frac{\sigma_t^2}{2}\Delta p_t$：由随机扩散造成的概率质量重新分布。
- $\Delta$：空间 Laplacian 算子。

该表达式假设噪声系数 $\sigma_t$ 只依赖时间，并且在各个空间维度上具有相同的强度。

#### 4.3 与 Continuity Equation 的比较

在 D1 中，我们介绍过连续性方程：

$$
\frac{\partial p_t(x)}{\partial t}
=
-\nabla\cdot\left(p_t(x)u_t(x)\right)
$$

它描述概率质量在速度场作用下的输运。

与之相比，Fokker–Planck 方程多出一项：

$$
\frac{\sigma_t^2}{2}\Delta p_t(x)
$$

这项描述随机扩散带来的变化。

当 $\sigma_t=0$ 时，Fokker–Planck 方程退化为连续性方程。

因此，可以将两者的区别总结为：

- **Continuity Equation：** 概率质量沿速度场输运。
- **Fokker–Planck Equation：** 概率质量既受到速度场输运，也受到随机扩散影响。

#### 4.4 为什么这一点对 Diffusion 很重要？

对于 SDE，即使知道漂移场 $u_t$，样本群体的概率密度也不只由漂移项决定，随机扩散项同样会影响分布。

这意味着：如果希望通过随机过程采样，同时又希望保持某条预先指定的概率路径，就需要协调漂移和扩散之间的关系。

这引出了下一节的 SDE Extension Trick。

---

### 5. SDE Extension Trick：加入随机性，同时保持目标概率路径

#### 5.1 核心思想

假设我们已经有一个目标概率路径 $p_t(x)$，并且它满足连续性方程：

$$
\frac{\partial p_t(x)}{\partial t}
=
-\nabla\cdot\left(p_t(x)u_t(x)\right)
$$

我们希望构造一个具有随机性的 SDE，但又让它的边缘概率密度仍然沿着这条目标路径演化。

课件给出的构造为：

$$
\boxed{
dX_t
=
\left[
u_t(X_t)
+
\frac{\sigma_t^2}{2}
\nabla\log p_t(X_t)
\right]dt
+
\sigma_t\,dW_t
}
$$

注意，这里漂移项不再只是 $u_t$，还增加了一个与 score 有关的修正项：

$$
\frac{\sigma_t^2}{2}\nabla\log p_t(X_t)
$$

#### 5.2 为什么要添加 Score Correction？

先定义修正后的漂移：

$$
b_t(x)
=
u_t(x)
+
\frac{\sigma_t^2}{2}\nabla\log p_t(x)
$$

代入 Fokker–Planck 方程：

$$
\frac{\partial p_t}{\partial t}
=
-\nabla\cdot(p_tb_t)
+
\frac{\sigma_t^2}{2}\Delta p_t
$$

展开漂移项：

$$
\begin{aligned}
\frac{\partial p_t}{\partial t}
={}&-\nabla\cdot(p_tu_t)\\
&-\nabla\cdot
\left(
\frac{\sigma_t^2}{2}
p_t\nabla\log p_t
\right)\\
&+\frac{\sigma_t^2}{2}\Delta p_t
\end{aligned}
$$

利用恒等式

$$
p_t\nabla\log p_t=\nabla p_t
$$

得到

$$
\frac{\partial p_t}{\partial t}
=
-\nabla\cdot(p_tu_t)
-\frac{\sigma_t^2}{2}\Delta p_t
+\frac{\sigma_t^2}{2}\Delta p_t
$$

两个扩散相关的项相互抵消，因此：

$$
\boxed{
\frac{\partial p_t}{\partial t}
=
-\nabla\cdot(p_tu_t)
}
$$

这正是原来的连续性方程。

以上推导假设 $\sigma_t$ 只依赖时间，并且密度与向量场满足所需的正则性条件。

#### 5.3 直觉理解

可以把整个过程理解为三部分：

1. $u_t$ 负责沿着目标速度场移动。
2. 随机噪声 $\sigma_t\,dW_t$ 使采样过程具有随机性。
3. Score correction 对漂移进行修正，抵消随机扩散在边缘密度演化中引入的额外变化。

因此，在适当条件下，可以构造一个**单个样本随机运动，但整体边缘分布仍遵循指定概率路径**的 SDE。

#### 5.4 使用 Score Network 进行随机采样

实际应用中，真实的 score 往往未知，可以用神经网络近似：

$$
s_t^\theta(x)\approx\nabla_x\log p_t(x)
$$

于是采样时可以采用近似的 SDE：

$$
dX_t
=
\left[
u_t(X_t)+\frac{\sigma_t^2}{2}s_t^\theta(X_t)
\right]dt
+
\sigma_t\,dW_t
$$

需要注意，使用近似 score 时，抵消关系不再保证精确成立；采样效果取决于 score 估计的准确性以及数值积分方法。

#### 5.5 Deterministic Sampling 与 Stochastic Sampling

| 比较维度 | 确定性采样 | 随机采样 |
|---|---|---|
| 典型形式 | ODE | SDE |
| 是否显式加入随机增量 | 否 | 是 |
| 主要动力学 | 速度场 | 漂移、扩散及其修正 |
| 本讲中的关键对象 | Vector Field | Score Function 与 SDE |

两种方法并非简单地对应“好”和“坏”。它们提供了不同的采样机制，实际表现取决于模型、概率路径和采样算法。

---

### 6. Classifier-Free Guidance（CFG）

#### 6.1 为什么需要 Guidance？

在文本条件生成中，我们希望模型不仅生成合理的图像，还能生成**符合指定提示词的图像**。

例如：

- 无条件提示：生成一张图像。
- 条件提示：生成一张猫正在烤蛋糕的图像。

即使生成模型能够生成自然的图像，也不代表每张图像都能准确遵循提示词。

Guidance 的目标是增强生成过程对条件信息的响应。

#### 6.2 CFG 的核心公式

记：

- $u_{\mathrm{uncond}}$：无条件预测的速度场。
- $u_{\mathrm{cond}}$：带有条件提示的速度场。
- $w$：Guidance Scale。

课件使用的 CFG 公式为

$$
\boxed{
u_{\mathrm{CFG}}
=
u_{\mathrm{uncond}}
+
w\left(
u_{\mathrm{cond}}-u_{\mathrm{uncond}}
\right)
}
$$

将公式展开：

$$
u_{\mathrm{CFG}}
=
(1-w)u_{\mathrm{uncond}}
+
wu_{\mathrm{cond}}
$$

其中

$$
u_{\mathrm{cond}}-u_{\mathrm{uncond}}
$$

表示条件提示带来的预测差异。

CFG 会沿着这个差异方向对预测进行调整，从而增强条件提示的影响。

#### 6.3 如何理解 Guidance Scale？

**当 $w=0$ 时：**

$$
u_{\mathrm{CFG}}=u_{\mathrm{uncond}}
$$

模型只使用无条件预测。

**当 $w=1$ 时：**

$$
u_{\mathrm{CFG}}=u_{\mathrm{cond}}
$$

得到普通的条件预测，没有额外的 CFG 放大。

**当 $w>1$ 时：**

$$
u_{\mathrm{CFG}}
=
u_{\mathrm{cond}}
+
(w-1)
\left(
u_{\mathrm{cond}}-u_{\mathrm{uncond}}
\right)
$$

此时，条件预测与无条件预测之间的差异被进一步放大。

直觉上，模型会更加积极地沿着有助于满足提示词的方向更新样本。

#### 6.4 CFG 训练：为什么需要空条件？

CFG 的关键设计是：同一个模型既要能处理有条件输入，也要能处理无条件输入。

训练时，随机丢弃一部分条件标签或提示词，将它们替换为空条件。

这样，模型就能学习两种预测：

$$
u_\theta(x_t,t,y)
$$

以及

$$
u_\theta(x_t,t,\varnothing)
$$

其中：

- $y$：正常的文本条件。
- $\varnothing$：空条件。

在采样阶段，通过分别计算条件预测和无条件预测，就可以组合得到 CFG 速度场：

$$
\begin{aligned}
u_{\mathrm{CFG}}
={}&u_\theta(x_t,t,\varnothing)\\
&+w\left[
u_\theta(x_t,t,y)
-u_\theta(x_t,t,\varnothing)
\right]
\end{aligned}
$$

这就是 Classifier-Free Guidance 名称的由来：它不需要额外训练一个独立的分类器来提供 guidance，而是利用同一生成模型的条件和无条件预测。

#### 6.5 Guidance Scale 越大越好吗？

不一定。

随着 $w$ 增大，模型可能更强烈地遵循提示词，但也可能牺牲图像的自然度、多样性或其他质量指标。

课件特别强调：CFG 是一种主要由经验效果支持的实用技巧。使用 $w>1$ 时，得到的速度场通常不再对应原始条件分布的普通预测。

因此，CFG 并不等价于严格地从原始数据分布采样。

**需要掌握的结论：**

- $w=1$：普通条件预测。
- $w>1$：放大条件预测与无条件预测之间的差异。
- 更强的 guidance 不保证整体生成质量一定更好。

---

### 7. 神经网络架构：U-Net 与 Diffusion Transformer

#### 7.1 生成模型对神经网络有什么要求？

本讲将生成模型中的神经网络理解为一个向量场或相关预测函数。

以条件图像生成为例，网络需要接收：

- 时间 $t$。
- 当前带噪图像或中间状态 $x_t$。
- 条件信息 $y$，例如文本提示。

网络输出通常需要与输入图像或潜变量具有相同的形状：

$$
u_\theta(x_t,t,y)\in\mathbb{R}^{\mathrm{shape}(x_t)}
$$

具体输出代表速度、噪声或其他等价的预测目标，取决于模型采用的参数化方式。

因此，普通的图像分类网络不能直接替代生成模型所需要的预测网络：分类器通常输出类别分数，而生成模型需要输出与当前状态同形状的张量。

#### 7.2 U-Net

U-Net 最初用于图像分割，之后被广泛用于 Diffusion Models。

其基本结构包含两部分：

- **Encoder：** 逐渐降低空间分辨率，同时增加通道数。
- **Decoder：** 逐渐恢复空间分辨率。
- **Skip Connections：** 将编码阶段的信息传递给对应的解码阶段。

这种 U 形结构既能提取较大范围的上下文，也能保留较精细的空间信息。

在生成模型中，还需要让网络利用时间 $t$ 和条件提示 $y$，例如通过时间嵌入、条件嵌入等机制。

#### 7.3 Diffusion Transformer（DiT）

DiT 的核心思路是将图像划分成多个 patches，并通过 Transformer 在这些 patches 之间进行注意力计算。

其基本思路来自 Vision Transformer（ViT）。

与 U-Net 的卷积结构不同，DiT 使用 Transformer 处理图像表示，并将时间与条件信息纳入生成网络。

可以从以下角度理解两者：

| 比较维度 | U-Net | DiT |
|---|---|---|
| 主要结构 | 卷积编码器—解码器 | Transformer |
| 主要处理方式 | 多尺度卷积特征 | Patch 表示与注意力 |
| 结构特点 | U 形结构及跳跃连接 | 在 token / patch 表示之间进行注意力计算 |
| 生成任务中的角色 | 预测速度、噪声等 | 同样预测速度、噪声等 |

两者都可以作为生成模型的骨干网络；选择哪一种取决于任务、计算资源和模型设计。

#### 7.4 Latent Diffusion：为什么在潜空间中生成？

直接在高分辨率图像上运行生成模型，计算成本很高；对于视频，额外的时间维度会进一步增加成本。

Latent Diffusion 的思路是：先使用预训练的（变分）自编码器，将图像编码到潜空间，再在潜变量上训练和运行生成模型。

概念上可以表示为：

$$
x \xrightarrow{\mathrm{Encoder}} z
$$

在潜空间中进行生成：

$$
z_T \longrightarrow z_0
$$

再通过解码器恢复图像：

$$
z \xrightarrow{\mathrm{Decoder}} x
$$

这里的时间方向取决于具体模型采用的约定；该示意图仅表达从噪声潜变量到目标潜变量的生成过程。

潜空间通常比原始图像空间具有更低的空间维度，因此能显著降低生成过程的计算开销。

**重点：** Latent Diffusion 改变的是生成发生的表示空间，不是 Flow Matching 或 Score Matching 的基本数学目标。

---

### 8. 大规模模型案例

#### 8.1 Stable Diffusion 3（SD3）

课件给出的 SD3 案例包含以下设计：

- 使用 Flow Matching。
- 采用 CondOT 风格的直线路径。
- 在预训练 VAE 的潜空间中进行生成。
- 使用 Classifier-Free Guidance。
- 使用 MM-DiT 架构。
- 课件示例规模约为 8B 参数。
- 课件示例采用约 50 个采样步骤。
- 课件提及 LAION 数据集作为数据示例。

其中，MM-DiT 是面向多模态信息设计的 Diffusion Transformer 架构。

可以将 SD3 的生成过程理解为：

1. 从潜空间噪声开始。
2. 将当前潜变量、时间与文本条件输入生成网络。
3. 预测当前状态的更新方向。
4. 根据采样算法逐步更新潜变量。
5. 通过 CFG 增强文本条件的影响。
6. 将生成的潜变量解码为图像。

这里的参数量和采样步数是课件案例中的数值，并不意味着所有配置或版本都具有完全相同的设置。

#### 8.2 Meta MovieGen

课件还介绍了 MovieGen 作为视频生成案例：

- 使用 Flow Matching。
- 使用 CondOT 风格的直线路径。
- 在潜空间中进行生成。
- 使用适配视频任务的 DiT 架构。
- 使用 Classifier-Free Guidance。
- 课件给出的模型规模约为 30B 参数。
- 课件提到训练使用 6,144 张 H100 GPU。

视频生成比静态图像生成更具挑战，因为模型不仅需要处理空间信息，还要处理时间维度上的连续性和一致性。

潜空间生成对于视频尤其重要：如果直接处理所有高分辨率视频帧，计算成本会非常高。

#### 8.3 这两个案例说明了什么？

大型生成模型不是单靠某一个技巧实现的，而是多种设计组合的结果：

- **Flow Matching：** 提供训练目标与生成动力学。
- **潜空间表示：** 降低生成计算的开销。
- **DiT / MM-DiT：** 处理复杂的图像或多模态信息。
- **CFG：** 增强生成结果对条件提示的响应。
- **数值采样：** 将模型预测转化为逐步生成的过程。

理解这些模块分别解决什么问题，比单纯记忆模型名称更重要。

---

### 9. Diffusion 文献地图：不同方法之间的关系

#### 9.1 不同论文可能使用不同的时间约定

阅读 Flow 和 Diffusion 文献时，一个常见困难是：不同方法对时间方向的定义并不一致。

**Flow Matching 常见约定：**

$$
t=0:\ \text{Noise}
\qquad
t=1:\ \text{Data}
$$

**Score-based Diffusion 的常见约定：**

$$
t=0:\ \text{Data}
\qquad
t\rightarrow\infty:\ \text{Noise}
$$

这两种时间约定方向相反。

因此，在比较公式之前，首先要确认：时间增加时，样本是在逐渐接近数据，还是逐渐变成噪声。

不要仅凭变量都叫 $t$，就认为两篇论文中的时间具有相同含义。

#### 9.2 连续时间与离散时间

课件还区分了连续时间和离散时间的建模方式。

连续时间方法可以使用 ODE 或 SDE 描述样本的演化。

离散时间方法则可以通过 Markov Chain 描述一系列随机状态之间的转移。

以经典方法为例：

- **DDPM：** 使用离散时间的扩散 Markov Chain。
- **DDIM：** 提供另一种采样方式，并与 Probability Flow ODE 有密切联系。

课件使用近似关系来概括 DDIM 与 Probability Flow ODE 的联系。更严格地说，具体关系取决于所采用的模型和采样设置，不应把所有 DDIM 过程都视为与任意 Probability Flow ODE 完全相同。

#### 9.3 不同的 Noising Procedures

不同生成方法还可能采用不同的方式构造从数据到噪声、或从噪声到数据的概率路径。

例如：

- Flow Matching / Rectified Flow：直接指定概率路径或插值过程。
- Denoising Diffusion Models：通常从一个逐步加噪的过程出发。
- Score-based Diffusion：学习 score，并通过对应的随机或确定性动力学进行采样。

这些方法的公式看起来可能不同，但在特定高斯概率路径及相应条件下，可以建立数学联系。

需要避免一个过度概括：**并不是任意两种扩散过程在所有设置下都完全等价。** 等价性需要依赖具体的路径、时间参数化和动力学条件。

#### 9.4 Flow Matching 为什么灵活？

课件强调，Flow Matching 可以用于将不同的源分布转换为不同的目标分布。

例如：

- 无音频视频 → 有音频视频。
- 低分辨率图像 → 高分辨率图像。
- 未受扰动的细胞 → 受扰动的细胞。

这些例子说明，Flow Matching 不只适用于传统的“噪声生成图像”。

从更一般的角度看，只要能够定义适当的源分布、目标分布以及连接两者的概率路径，就可以考虑学习对应的速度场。

这也是 Flow Matching 在生成建模之外具有应用潜力的原因。

---

### 10. 全讲总结：把数学概念连接起来

#### 10.1 需要建立的概念关系

**第一层：概率路径**

$$
p_t(x)
$$

描述不同时间的样本分布。

**第二层：Vector Field**

$$
u_t(x)
$$

描述样本沿着怎样的方向移动，并通过连续性方程描述概率质量的输运。

**第三层：Score Function**

$$
s_t(x)=\nabla_x\log p_t(x)
$$

描述概率密度的对数梯度。在适当的高斯概率路径下，它可以与速度场相互转换。

**第四层：随机动力学**

$$
dX_t=u_t(X_t)\,dt+\sigma_t\,dW_t
$$

通过 SDE 引入随机性；Fokker–Planck 方程描述相应的边缘密度演化。

**第五层：采样修正**

$$
u_t(x)+\frac{\sigma_t^2}{2}s_t(x)
$$

在本讲讨论的条件下，利用 score 修正漂移，使加入随机噪声的 SDE 仍可沿着目标概率路径演化。

**第六层：条件控制**

$$
u_{\mathrm{CFG}}
=
u_{\mathrm{uncond}}
+
w(u_{\mathrm{cond}}-u_{\mathrm{uncond}})
$$

CFG 通过放大条件预测与无条件预测之间的差异，提高生成过程对条件提示的响应。

#### 10.2 最值得掌握的公式

建议优先熟悉下面六个公式。

**1. Score Function**

$$
s_t(x)=\nabla_x\log p_t(x)
$$

**2. CondOT 线性路径**

$$
x_t=(1-t)\epsilon+tz
$$

**3. CondOT 条件速度**

$$
u_t(x_t\mid z)=z-\epsilon
$$

**4. CondOT 条件 Score**

$$
s_t(x_t\mid z)=-\frac{\epsilon}{1-t}
$$

**5. Fokker–Planck Equation**

$$
\frac{\partial p_t}{\partial t}
=
-\nabla\cdot(p_tu_t)
+
\frac{\sigma_t^2}{2}\Delta p_t
$$

**6. Classifier-Free Guidance**

$$
u_{\mathrm{CFG}}
=
u_{\mathrm{uncond}}
+
w(u_{\mathrm{cond}}-u_{\mathrm{uncond}})
$$

#### 10.3 学习优先级

**优先级 A：需要能够推导**

- 从高斯分布的对数密度推导 score。
- 从 CondOT 线性路径推导速度和条件 score。
- 理解 DSM 为什么可以通过噪声预测学习 score。
- 从 Fokker–Planck 方程理解 SDE Extension Trick 中的抵消关系。
- 理解 CFG 公式在 $w=0$、$w=1$ 和 $w>1$ 时的含义。

**优先级 B：需要能够解释**

- Conditional 与 Marginal 的区别。
- 为什么 Score Matching 和 Flow Matching 存在联系。
- 为什么潜空间生成能够降低计算成本。
- U-Net 与 DiT 的基本架构区别。
- CFG 为什么可能提高提示词一致性，却不保证生成质量单调提升。

**优先级 C：建立全局认知**

- DDPM、DDIM、Score-based Diffusion 与 Flow Matching 的关系。
- SD3 与 MovieGen 如何组合多种设计。
- Flow Matching 如何扩展到不同源分布与目标分布之间的转换。

---

### 11. 自测题

#### 11.1 概念理解

1. Score Function 为什么定义为 $\nabla_x\log p_t(x)$，而不是直接使用 $p_t(x)$？
2. Conditional Score 和 Marginal Score 有什么区别？
3. 为什么在 CondOT 路径中，条件速度 $z-\epsilon$ 不依赖于时间 $t$？
4. Fokker–Planck 方程相比连续性方程多出的项是什么？它代表什么？
5. 为什么 SDE Extension Trick 需要在漂移项中加入 score correction？
6. 为什么 CFG 训练时需要随机丢弃一部分条件提示？
7. 为什么 CFG 的 guidance scale 不是越大越好？
8. 为什么 Latent Diffusion 对视频生成尤其有用？

#### 11.2 推导练习

**练习 1：标准高斯的 Score**

已知

$$
p(x)=\mathcal{N}(x;0,I)
$$

请从对数概率密度出发，推导

$$
\nabla_x\log p(x)=-x
$$

**练习 2：CondOT 速度**

已知

$$
x_t=(1-t)\epsilon+tz
$$

对时间 $t$ 求导，并解释为什么得到的结果可以作为条件目标速度。

**练习 3：CFG**

已知

$$
u_{\mathrm{CFG}}
=
u_{\mathrm{uncond}}
+
w(u_{\mathrm{cond}}-u_{\mathrm{uncond}})
$$

分别代入 $w=0$、$w=1$ 和 $w=2$，解释三种情况下的预测结果。

**练习 4：SDE Extension Trick**

从

$$
p_t\nabla\log p_t=\nabla p_t
$$

出发，解释为什么 score correction 引入的散度项可以与 Fokker–Planck 方程中的扩散项抵消。

---

### 12. 一句话总结

D2 的主线是：**Flow Matching 学习速度场，Score Matching 学习对数概率密度的梯度；对于适当的高斯概率路径，两者可以相互联系。利用 score，还能构造与目标概率路径相容的随机采样过程。CFG 进一步控制生成结果对提示词的响应，而 U-Net、DiT 和潜空间生成则为这些数学方法提供了可扩展的工程实现。**